# EEG-Based Prosthetic Arm Motor-Imagery Data Collection Paradigm

This notebook runs a visual/audio stimulus paradigm used to cue a participant through **open-hand** and **close-hand** motor (or motor-imagery) actions while EEG/EMG is recorded by separate acquisition hardware. Each trial's stimulus timings are logged and rendered as a Gantt-style timeline for post-hoc alignment with the recorded signal.

**How it's organized below:**
1. Imports
2. `collect_data()` — placeholder hook for the external data-acquisition trigger
3. `show_image()` — displays a fullscreen stimulus image, optionally with audio
4. `plot_gantt_with_images()` — visualizes a trial's stimulus timeline
5. `main()` — runs the full 5-trial session
6. Run cell

> **Before running:** this notebook expects the stimulus images/audio in an `assets/` folder next to it (see the README for the expected file names). Update the `assets/` paths below if your files live elsewhere.

In [ ]:
import time
import cv2
import pygame
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd

## Data collection hook

Placeholder for whatever triggers/synchronizes your EEG/EMG acquisition at the start of each trial. Replace the body with a call into your acquisition SDK or a hardware trigger (e.g. LSL marker, serial pulse).

In [ ]:
def collect_data():
    print("Data collection in progress...")
    time.sleep(1)  # Simulated time for data collection
    print("Data collection completed.")

## Stimulus display

Shows a stimulus image fullscreen for a fixed duration, optionally playing an audio cue alongside it.

In [ ]:
def show_image(img_path, audio_path=None, duration=2000):
    if audio_path:
        pygame.mixer.music.load(audio_path)
        pygame.mixer.music.play()

    img = cv2.imread(img_path)
    if img is None:
        print(f"Error: could not load image {img_path}")
        return

    window_name = "Stimulus"

    # Create fullscreen window
    cv2.namedWindow(window_name, cv2.WND_PROP_FULLSCREEN)
    cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

    cv2.imshow(window_name, img)
    cv2.waitKey(duration)
    cv2.destroyAllWindows()

## Trial timeline visualization

Builds a Gantt-style chart of a trial's stimulus sequence, with the corresponding stimulus image thumbnailed onto each segment. Trial 1 uses a longer pre-paradigm/countdown sequence; trials 2-5 use a shorter open/relax/close/relax sequence.

In [ ]:
def plot_gantt_with_images(iteration_num, is_first_trial=False):
    # Define durations for each step
    if is_first_trial:
        durations = {
            "ST_Trial_PreParadigm": 6,
            "Countdown_3": 1,
            "Countdown_2": 1,
            "Countdown_1": 1,
            "ST_Trial_Start": 1,
            "ST_Trial_Open_6s": 6,
            "ST_Trial_Close_6s": 6,
            "ST_Trial_PostParadigm": 5
        }
        order = [
            "ST_Trial_PreParadigm", "Countdown_3", "Countdown_2", "Countdown_1",
            "ST_Trial_Start", "ST_Trial_Open_6s", "ST_Trial_Close_6s", "ST_Trial_PostParadigm"
        ]
    else:
        durations = {
            "ST_Trial_PreParadigm": 0,
            "ST_Trial_Open_6s": 6,
            "Relax1": 1,
            "ST_Trial_Close_6s": 6,
            "Relax2": 5,
            "ST_Trial_PostParadigm": 0
        }
        order = [
            "ST_Trial_PreParadigm", "ST_Trial_Open_6s", "Relax1",
            "ST_Trial_Close_6s", "Relax2", "ST_Trial_PostParadigm"
        ]

    image_map = {
        "ST_Trial_PreParadigm": "assets/plus_sign.png",
        "Countdown_3": "assets/plus_sign.png",
        "Countdown_2": "assets/plus_sign.png",
        "Countdown_1": "assets/plus_sign.png",
        "ST_Trial_Start": "assets/plus_sign.png",
        "ST_Trial_Open_6s": "assets/openhand.jpg",
        "Relax1": "assets/plus_sign.png",
        "ST_Trial_Close_6s": "assets/closehand.jpg",
        "Relax2": "assets/plus_sign.png",
        "ST_Trial_PostParadigm": "assets/plus_sign.png",
    }

    # Build the timeline per task
    start_time = 0
    tasks = []

    for step in order:
        duration = durations[step]
        tasks.append({
            "Task ID": f"Trial {iteration_num + 1} - {step}",
            "Start Time": start_time,
            "End Time": start_time + duration,
            "Label": step
        })
        start_time += duration

    df = pd.DataFrame(tasks)

    fig, ax = plt.subplots(figsize=(12, 4))

    for i, row in df.iterrows():
        label = row["Label"]
        img_path = image_map.get(label)

        if img_path:
            try:
                img = mpimg.imread(img_path)
                center_time = (row["Start Time"] + row["End Time"]) / 2
                ax.imshow(img, extent=(center_time - 1, center_time + 1,
                                        i - 0.4, i + 0.4),
                          aspect='auto', zorder=10)
            except FileNotFoundError:
                print(f"Image not found: {img_path}")

    # Axis config
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Task ID")
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df["Task ID"])
    ax.set_xlim(0, df["End Time"].max() + 1)
    ax.set_title(f"EEG Timeline with Images - Trial {iteration_num + 1}")
    plt.tight_layout()
    plt.show()

## Experiment session

Runs 5 trials end-to-end: triggers data collection, steps through the stimulus sequence for that trial, logs stimulus onset times, plots the trial timeline, then rests before the next trial. Plays a closing audio/image cue at the end of the session.

In [ ]:
def main():
    pygame.init()

    # Image/audio sequence used for Trials 2-5 (Open, Relax1, Close, Relax2)
    images = [
        ("Open", "assets/openhand.jpg", "assets/open.mp3", 6000),
        ("Relax1", "assets/plus_sign.png", None, 1000),
        ("Close", "assets/closehand.jpg", "assets/close.mp3", 6000),
        ("Relax2", "assets/plus_sign.png", None, 5000),
    ]

    thanks_image = "assets/thanks.png"
    thanks_audio = "assets/thanks.mp3"

    for iteration in range(5):
        logs = []
        t0 = time.time()

        print(f"\nStarting Trial {iteration + 1}")
        collect_data()

        is_first_trial = iteration == 0

        # Show images with recording based on the trial
        if is_first_trial:
            # Handle Trial 1 separately (28 seconds total)
            trial_images = [
                ("ST_Trial_PreParadigm", "assets/plus_sign.png", None, 6000),
                ("Countdown_3", "assets/plus_sign.png", None, 1000),
                ("Countdown_2", "assets/plus_sign.png", None, 1000),
                ("Countdown_1", "assets/plus_sign.png", None, 1000),
                ("ST_Trial_Start", "assets/plus_sign.png", None, 1000),
                ("ST_Trial_Open_6s", "assets/openhand.jpg", "assets/close.mp3", 6000),
                ("ST_Trial_Close_6s", "assets/closehand.jpg", "assets/close.mp3", 6000),
                ("ST_Trial_PostParadigm", "assets/plus_sign.png", None, 5000),
            ]
        else:
            # Use the standard sequence for Trials 2-5
            trial_images = images

        # Show images with recording
        for label, img_path, audio_path, duration_ms in trial_images:
            show_image(img_path, audio_path, duration_ms)
            logs.append({
                "Label": label,
                "Time (s)": round(time.time() - t0, 2)
            })

        # Plot timeline for this trial
        logs_df = pd.DataFrame(logs)
        plot_gantt_with_images(iteration, is_first_trial)

        # Break before next trial
        print("Resting 5 seconds before next trial...\n")
        time.sleep(5)

    # End message
    pygame.mixer.music.load(thanks_audio)
    pygame.mixer.music.play()
    cv2.imshow("Image", cv2.imread(thanks_image))
    cv2.waitKey(7000)
    cv2.destroyAllWindows()
    print("Procedure completed under guidance.")

## Run the session

This opens fullscreen OpenCV windows and plays audio, so it needs a display and audio device — run it as a local script or a notebook kernel on the machine connected to the stimulus monitor, not in a headless/remote environment.

In [ ]:
if __name__ == "__main__":
    main()